# Cleaning the Clinical Data:
The purpose of this notebook is to clean the multi-center clinical data. As the steps of cleaning were too specific for this dataset, a notebook is dedicated to it instead of a generic script, to explain the decisions taken. The final dataset includes the cleaned selected features and target columns for each research question. The target columns are as below:

For treatment response: **surv_bestresponse_car**  
For ICANS toxicity: **ae_summ_icans_highestgrade_v2 and ae_summ_icans_v2**  
For CRS toxicity: **ae_summ_crs_v2, ae_summ_highestgrade_v2**  
## Cleaning Steps:

### Specifying the clinical feature subset
Not all the features are relevant for the analysis. The selection of features are clinically driven and are recorded before the pre-LD phase.

### Duplicate rows handeling
Some of the patients' data are duplicated in the clinical dataset, we need to remove them first.

### Intersect with the Radiomics dataset
We only want clinical data for which we have the radiomics features. **Note that patient umcg 95 is present in radiomics but not in clinical.**  

### Fix the clinical dataset inconsistancy
There are inconsistencies in how a Nan value is defined. Also some of the numeric values are wrongly formatted which would make it impossible to cast a numeric value on them.

### One Hot Encoding
The indication_dis_diagnosis which specifies the type of lymphoma is not one hot encoded originally.
 
### Type casting
all the features should be numerical, except for ID.  

### Handling missing values 
1. total_num_priortherapylines_fl: fill with 0, as it means they did not receive the therapy as it's a sign for Not Applicable.    
2. indication_extranodal_nr: fill based on indication_extran_invol, as it's only Nan for when the involvment is either 0 or 1.    
3. cli_st_neutrophils: only one missing, use median imputation, it's safe.    
4. ae_summ_icans_highestgrade_v2: fill with 0, the missing values are for when patients have not shown ICANS toxicity.    
5. indication_sec_refr: filled with "2", as in the original clinical data, the Nans are a group, encoded with 2, this one instance is not coded.  
6. indication_whops: only 1 missing, filled with median as it's safe.  
7. for cli_st_ldh (2%), cli_st_crp (10%) and cli_st_ferritin (31%), median is still a viable imputation method as there's not a large precentage of data missing.  
8. ae_summ_highestgrade_v2: fill based on ae_summ_crs_v2, if the value is Nan, it means the patient has not experienced any CRS toxicity and the ae_summ_crs_v2 is 0.
9. surv_bestresponse_car: fill with 0. The missing values are patients who passed away before the treatment response could be assessed. They will be assigned their own group, encoded as 0.  
### Dropping columns with zero variance
These columns add no information, so it's better if we drop them.

In [18]:
import yaml
import pandas as pd

with open('config.yaml', 'r') as file:
    config = yaml.safe_load(file)

In [19]:
radiomics_delta = pd.read_csv(config['delta']['combined'])
clinical = pd.read_csv(config['clinical']['combined'])

In [ ]:
# Specifying the clinical feature subset
clinical_ftrs = [
    "record_id",	
    "scr_sex",
    "scr_age",
    "scr_height",
    "scr_weight",
    "scr_bmi",
    "indication_dis_diagnosis.factor",
    "total_num_priortherapylines_fl",
    "total_num_priortherapylines_aggressive",
    "indication_priorsct",
    "indication_whops",
    "indication_ldh_uln",
    "indication_age_60",
    "indication_bulkydisease",
    "indication_stage",
    "indication_extran_invol",
    "indication_extranodal_nr",
    "indication_pri_refr",
    "indication_sec_refr",
    "indication_dis_lymsubtype_cns",
    "tr_car_br",
    "tr_car_bridg_reg___1",
    "tr_car_bridg_reg___2",
    "tr_car_bridg_reg___3",
    "tr_car_bridg_reg___4",
    "tr_car_bridg_reg___5",
    "tr_car_bridg_reg___6",
    "tr_car_bridg_reg___7",
    "tr_car_bridg_reg___8",
    "tr_car_bridg_reg___9",
    "tr_car_bridg_reg___10",
    "tr_car_bridg_reg___11",
    "tr_car_bridg_reg___12",
    "tr_car_bridg_reg___na",
    "tr_car_bridg_reg___ne",
    "cli_st_hemoglobin",
    "cli_st_trombocytes",
    "cli_st_leukocytes",
    "cli_st_neutrophils",
    "cli_st_ldh",
    "cli_st_crp",
    "cli_st_ferritin",
    "ae_summ_icans_highestgrade_v2",
    "ae_summ_icans_v2",
    "ae_summ_crs_v2",
    "ae_summ_highestgrade_v2",
    "surv_bestresponse_car"

]

In [21]:
clinical_filtered = clinical.loc[1: , clinical_ftrs]

In [22]:
# Dropping duplicates in the 'record_id' column
clinical_filtered.drop_duplicates(subset='record_id', inplace=True)

In [23]:
# Intersect with the Radiomics dataset
clinical_filtered = clinical_filtered[clinical_filtered['record_id'].isin(radiomics_delta['Unnamed: 0'])]

In [24]:
clinical_filtered.reset_index(drop=True, inplace=True)

In [25]:
# Fix the clinical dataset inconsistancies
clinical_filtered = clinical_filtered.replace(['NE','ne', 'NA','Not evaluated'], pd.NA, inplace=True)
clinical_filtered = clinical_filtered.replace('6,4', '6.4', inplace=True)
clinical_filtered = clinical_filtered.replace('5,0 ', '5.0', inplace=True)
clinical_filtered = clinical_filtered.replace('<0.5', '0.25', inplace=True) # Assuming <0.5 means 0.25 for the sake of numerical analysis

In [26]:
# One Hot Encoding the 'indication_dis_diagnosis' column
clinical_filtered = pd.get_dummies(clinical_filtered, columns=['indication_dis_diagnosis.factor'], drop_first=False)

In [27]:
# Convert only numeric-expected columns
num_cols = [
    c for c in clinical_filtered.columns
    if c != "record_id"
]

clinical_filtered[num_cols] = clinical_filtered[num_cols].apply(
    pd.to_numeric, errors="raise"
).astype("float64")

In [28]:
# Handling missing values
clinical_filtered['total_num_priortherapylines_fl'] = clinical_filtered['total_num_priortherapylines_fl'].fillna(0)
clinical_filtered['indication_extranodal_nr'] = clinical_filtered['indication_extranodal_nr'].fillna(clinical_filtered['indication_extran_invol'])
clinical_filtered['ae_summ_icans_highestgrade_v2'] = clinical_filtered['ae_summ_icans_highestgrade_v2'].fillna(0)
clinical_filtered['indication_sec_refr'] = clinical_filtered['indication_sec_refr'].fillna(2)
clinical_filtered[['indication_whops', 'cli_st_ldh','cli_st_crp','cli_st_ferritin', 'cli_st_neutrophils']] = \
    clinical_filtered[['indication_whops', 'cli_st_ldh','cli_st_crp','cli_st_ferritin', 'cli_st_neutrophils']].fillna(clinical_filtered[['indication_whops', 'cli_st_ldh','cli_st_crp','cli_st_ferritin', 'cli_st_neutrophils']].median())
clinical_filtered['ae_summ_highestgrade_v2'] = clinical_filtered['ae_summ_highestgrade_v2'].fillna(clinical_filtered['ae_summ_crs_v2'])
clinical_filtered['surv_bestresponse_car'] = clinical_filtered['surv_bestresponse_car'].fillna(0)

In [29]:
clinical_filtered.isna().sum().sum()

np.int64(0)

In [30]:
clinical_filtered.info()

<class 'pandas.DataFrame'>
RangeIndex: 85 entries, 0 to 84
Data columns (total 51 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   record_id                                    85 non-null     str    
 1   scr_sex                                      85 non-null     float64
 2   scr_age                                      85 non-null     float64
 3   scr_height                                   85 non-null     float64
 4   scr_weight                                   85 non-null     float64
 5   scr_bmi                                      85 non-null     float64
 6   total_num_priortherapylines_fl               85 non-null     float64
 7   total_num_priortherapylines_aggressive       85 non-null     float64
 8   indication_priorsct                          85 non-null     float64
 9   indication_whops                             85 non-null     float64
 10  indication_ldh_

In [31]:
clinical_filtered.head(3)

,record_id,scr_sex,scr_age,scr_height,scr_weight,scr_bmi,total_num_priortherapylines_fl,total_num_priortherapylines_aggressive,indication_priorsct,indication_whops,...,ae_summ_icans_highestgrade_v2,ae_summ_icans_v2,ae_summ_crs_v2,ae_summ_highestgrade_v2,surv_bestresponse_car,indication_dis_diagnosis.factor_DLBCL,indication_dis_diagnosis.factor_HGBCL DH/TH,indication_dis_diagnosis.factor_HGBCL NOS,indication_dis_diagnosis.factor_PMBCL,indication_dis_diagnosis.factor_tFL
0,FTC-EMC-0022,0.0,53.0,183.0,100.0,30.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,FTC-EMC-0023,1.0,32.0,168.0,60.0,21.0,0.0,2.0,4.0,1.0,...,4.0,1.0,1.0,2.0,2.0,0.0,0.0,0.0,1.0,0.0
2,FTC-EMC-0027,0.0,69.0,167.0,71.0,25.0,0.0,2.0,4.0,1.0,...,1.0,1.0,1.0,2.0,4.0,1.0,0.0,0.0,0.0,0.0


In [32]:
# Finding and dropping zero variance columns
zero_var_cols = clinical_filtered[num_cols].var() == 0
clinical_filtered.drop(zero_var_cols.index[zero_var_cols], axis=1, inplace=True)

In [33]:
# save the cleaned clinical dataset
clinical_filtered.to_csv(config['clinical']['cleaned'], index=False)